<a href="https://colab.research.google.com/github/ZIZOUTOUMOU/LAB02-AdvDB/blob/main/Copy_of_iot_pw2_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob, os, pickle, time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [ ]:
csv_path = None
local = glob.glob("**/smoke_detection_iot.csv", recursive=True)
if local:
    csv_path = local[0]
else:
    try:
        import kagglehub
        folder = kagglehub.dataset_download("deepcontractor/smoke-detection-dataset")
        csv_path = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)[0]
    except Exception as e:
        print("Kaggle download failed:", e)
        from google.colab import files
        up = files.upload()
        csv_path = list(up.keys())[0]

df = pd.read_csv(csv_path)
print(csv_path, df.shape)
df.head()

100%|██████████| 1.64M/1.64M [00:01<00:00, 1.38MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/deepcontractor/smoke-detection-dataset/versions/1/smoke_detection_iot.csv (62630, 16)


,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [ ]:
drop_cols = [c for c in ["Unnamed: 0", "UTC", "CNT"] if c in df.columns]
X = df.drop(columns=drop_cols + ["Fire Alarm"])
y = df["Fire Alarm"]
print("Features:", list(X.columns))
print("Class balance:\n", y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

Features: ['Temperature[C]', 'Humidity[%]', 'TVOC[ppb]', 'eCO2[ppm]', 'Raw H2', 'Raw Ethanol', 'Pressure[hPa]', 'PM1.0', 'PM2.5', 'NC0.5', 'NC1.0', 'NC2.5']
Class balance:
 Fire Alarm
1    44757
0    17873
Name: count, dtype: int64
(50104, 12) (12526, 12)


In [ ]:
pipelines = {
    "LR-Pipeline": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "XGB-Pipeline": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(eval_metric="logloss", random_state=42)),
    ]),
}

results = {}
for name, pipe in pipelines.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    results[name] = {"train": time.perf_counter() - t0}
    y_pred = pipe.predict(X_test)
    results[name]["acc"] = accuracy_score(y_test, y_pred)
    results[name]["f1"] = f1_score(y_test, y_pred)

print(f"{'Model':<14}{'Accuracy':>10}{'F1':>10}{'Train (s)':>12}")
for n, r in results.items():
    print(f"{n:<14}{r['acc']:>10.4f}{r['f1']:>10.4f}{r['train']:>12.4f}")

Model           Accuracy        F1   Train (s)
LR-Pipeline       0.8951    0.9281      0.8789
XGB-Pipeline      0.9999    0.9999      0.7958


In [ ]:
for name, pipe in pipelines.items():
    fname = f"{name}.pkl"
    with open(fname, "wb") as f:
        pickle.dump(pipe, f)
    results[name]["size"] = os.path.getsize(fname) / 1024

size_table = pd.DataFrame({"Size (KB)": {n: round(r["size"], 2) for n, r in results.items()}})
size_table

,Size (KB)
LR-Pipeline,1.65
XGB-Pipeline,125.24


In [ ]:
for name, pipe in pipelines.items():
    runs = []
    for _ in range(5):
        t0 = time.perf_counter()
        pipe.predict(X_test)
        runs.append(time.perf_counter() - t0)
    total = min(runs)
    results[name]["total_inf"] = total
    results[name]["single"] = total / len(X_test) * 1000

time_table = pd.DataFrame({
    "Total Test Inference Time (s)": {n: round(r["total_inf"], 6) for n, r in results.items()},
    "Single Inference Time (ms)":    {n: round(r["single"], 6) for n, r in results.items()},
})
time_table

,Total Test Inference Time (s),Single Inference Time (ms)
LR-Pipeline,0.002160,0.000172
XGB-Pipeline,0.016971,0.001355
